<img src="https://github.com/Ch3ddarr/DSB-Unit-03/blob/main/assets/ga-logo.png?raw=1" style="float: left; margin: 20px; height: 55px">

# Introduction to Multiple Linear Regression

---

### About
An introduction to Linear Regression: what it is, how it works, and how to run a model using `scikit-learn`.

### Learning Objectives
- Describe modeling.
- Calculate mean squared error.
- State the assumptions of a linear regression model.
- Be able to interpret the coefficients of a linear regression model.
- Identify the difference between simple and multiple linear regression.
- Fit, generate predictions from, and evaluate a linear regression model in `sklearn`, and maybe in `statsmodels`.

### Notebook Guide

Part II. **Multiple Linear Regression**
- Introduction to MLR Model (see slides)
- Fitting an MLR model in `sklearn`
- Assumptions of MLR Model
- NOW YOU TRY
- Additional, not-really-BONUS topics:
    1. Multicollinearity
    1. The Curse of Dimensionality
    1. Fitting an MLR model in `statsmodels`
    1. Does the Scale of the Features Matter?
- Conclusions and Takeaways

In [74]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn import metrics

# Part II: Multiple Linear Regression

When you have more than 1 predictor variable you are doing _multiple linear regression_.

## Bikeshare Data 🚴‍♂️
Data adapted from source: [here](https://archive.ics.uci.edu/ml/datasets/bike+sharing+dataset)


Multiple Linear Regression (MLR) with Categorical Features

#### Goal: predict the number of riders (count), based on the features.

In [102]:
bike = pd.read_csv("https://raw.githubusercontent.com/Ch3ddarr/DSB-Unit-03/refs/heads/main/data/bikeshare.csv")
print(bike.shape)
bike.head()

(10886, 12)


,datetime,season,holiday,workingday,weather,temp,atemp,humidity,windspeed,casual,registered,count
0,2011-01-01 00:00:00,1,0,0,1,9.84,14.395,81,0.0,3,13,16
1,2011-01-01 01:00:00,1,0,0,1,9.02,13.635,80,0.0,8,32,40
2,2011-01-01 02:00:00,1,0,0,1,9.02,13.635,80,0.0,5,27,32
3,2011-01-01 03:00:00,1,0,0,1,9.84,14.395,75,0.0,3,10,13
4,2011-01-01 04:00:00,1,0,0,1,9.84,14.395,75,0.0,0,1,1


In [103]:
bike['season'].unique()

array([1, 2, 3, 4])

### Some preprocessing to clean things up:
1. Rename "count" to "riders"
2. Drop "casual" and "registered"
3. Cast "datetime" to actually be a datetime
4. Drop rows where weather is 4

In [101]:
# Some preprocessing:
# 1 - Rename "count" to "riders"
# 2 - Drop "casual" and "registered"
# 3 - Cast "datetime" to actually be a datetime
# 4 - Drop only row where weather == 4
bike.rename(columns={'count':'riders'}, inplace=True)
bike.drop(columns=['casual', 'registered'], inplace=True)

# Datetime will be your friend every time you deal with dates
# You do not want dates as object or strings
bike['datetime'] = pd.to_datetime(bike['datetime'])
# bike = bike[bike['weather'] != 4]


KeyError: "['casual', 'registered'] not found in axis"

In [104]:
bike['hour'] == bike['datetime'].dt.hour
xvars = ["season", "holiday", "workingday", "weather", "temp", "humidity", "windspeed", "hour"]

X = bike[xvars]
X = pd.get_dummies(columns=["season", "weather", "hour"], data=X, drop_first=True)

KeyError: 'hour'

In [78]:
# Season vs Month
pd.crosstab(bike['season'], bike['datetime'].dt.month)

datetime,1,2,3,4,5,6,7,8,9,10,11,12
season,,,,,,,,,,,,
1,884,901,901,0,0,0,0,0,0,0,0,0
2,0,0,0,909,912,912,0,0,0,0,0,0
3,0,0,0,0,0,0,912,912,909,0,0,0
4,0,0,0,0,0,0,0,0,0,911,911,912


### Ridership model
Let's try to predict the number of riders based on temperature and what season it is. We'll have to **one-hot encode** the season column!

### Step 1: Assemble our predictor variables (X) and our target (y)

In [79]:
# Dummify "season"
X = bike[["season", "temp"]]

# X is the predictors: the input variables for your model
# Get dummies is the function in Pandas that has...
X = pd.get_dummies(columns=['season'], drop_first = True, data=X, dtype=int)

y = bike['riders']

In [80]:
X.tail()

,temp,season_2,season_3,season_4
10881,15.58,0,0,1
10882,14.76,0,0,1
10883,13.94,0,0,1
10884,13.94,0,0,1
10885,13.12,0,0,1


In [81]:
y

,riders
0,16
1,40
2,32
3,13
4,1
...,...
10881,336
10882,241
10883,168
10884,129


### Step 2: Instantiate the model

In [82]:
# Here you choose the model
model = LinearRegression()

### Step 3: Fit the model

In [83]:
# Fit model
model.fit(X,y)

# This step is where the data is presented to the model and statistical
# calculations are done to minimize errors and select the best parameters
# of the model

LinearRegression()

### Step 4: Check out and interpret our coefficients

In [84]:
# Coefficients : another word for parameters in a linear model
model.coef_

array([  13.74082452,  -42.52608489, -105.33297379,   26.05004381])

In [85]:
set(zip(X.columns, model.coef_))

{('season_2', np.float64(-42.5260848879288)),
 ('season_3', np.float64(-105.33297379112757)),
 ('season_4', np.float64(26.0500438062949)),
 ('temp', np.float64(13.740824516865292))}

#### Interpretation of coefficients
Now that we have multiple variables, our interpretation are **holding all other variables constant**.
> For every 1 unit increase in $x_i$, we expect $y$ to increase by $\beta_i$, holding all else constant.

Furthermore, each interpretation of a dummy variable is **in relation to the baseline** (in this case, Winter).

**Interpretation of $\hat{\beta}_3$ (`season_3` - Summer):**
Holding the temperature constant, the effect of it being Summer contributes to 105 fewer riders per hour relative to Winter.

<details><summary>Interpret the coefficient for temp.</summary>
Holding all else constant, for every one-degree increase in temperature, we expect about 14 more riders per hour.
</details>

<details><summary>Interpret the coefficient for season_4 (Fall).</summary>
Holding the temperature constant, the effect of it being Fall contributes to 26 more riders per hour relative to Winter.
</details>

### Step 5: Make predictions

In [86]:
y_preds = model.predict(X)
y_preds

array([ 79.37369065,  68.10621455,  68.10621455, ..., 161.76111498,
       161.76111498, 150.49363888])

### Step 6: Evaluate the model

In [87]:
# R2:
# Y: is the ground truth
# Y_preds are the values predicted with your model
metrics.r2_score(y,y_preds)

0.19237940075618343

In [88]:
# MSE:
metrics.mean_squared_error(y, y_preds)

26498.273247869267

In [99]:
# RMSE:
metrics.root_mean_squared_error(y, y_preds)

162.78290219758728

<details><summary>How is our model?</summary>
Overall, this model isn't great. Only about 20% of the variation in ridership can be explained by the model.
</details>

So our model is pretty bad. How much should we trust our interpretation for the coefficients?
> Do our LINE assumptions hold?

When working with multiple linear regression, we add a secret 5th "assumption" to our LINE assumptions that is usually violated at least a little.

### LINEM
- L - Linear relationship
- I - Independent errors
- N - Normally distributed errors
- E - Equal variance of errors (homoscedasticity)
- M - No Multicollinearity/Independence of Predictors

In [90]:
# L:


In [91]:
# I:
# assumed
# although this is probably violated since this is time-series data

In [92]:
# N:


In [93]:
# E:


In [94]:
# M/I:
# Honestly temperature might be a little correlated with season, but not enough to be worried


Yikes! We should probably not say anything very strong about the coefficients in our interpretation.

# NOW YOU TRY

Let's make a better model.

Now your turn: Fit a model that uses _all_ x-variables (excluding `atemp`) as well as the **the hour of day**. One-hot encode columns accordingly, and check the assumptions.

In [95]:
# E


# Additional, not-really-BONUS topics

## ANRBT #1: Multicollinearity
Multicollinearity occurs when you have two or more $x$-variables that are correlated with each other. This phenomenon doesn't affect the quality of the model fit, but it completely destroys model inference (i.e., confidence intervals and hypothesis tests). In addition, it can also contribute to a **rank-deficient matrix**, meaning the model can't be fit at all!

## ANRBT #2: The Curse of Dimensionality
It turns out that the more variables we add to our model, the better it is. Actually, it doesn't even matter what this variable was, that will always be true. Even if it's completely irrelevant. That is, if we added the following variables to our model:

* Is there a sunspot on this day?
* The closing stock price of Apple that day.
* Whether or not Guy Fieri had pancakes for breakfast that day.
* A column of completely random standard normal variables

Our model is **guaranteed** to get better. But it isn't _really_ a better model, right? It turns out, we're modeling **noise**, not **signal**. We'll talk more about this phenomenon later.

## ANRBT #3: The `statsmodels` Library
`scikit-Learn` does not hold the monopoly in doing linear regression in Python. In fact, you may even like this better!

## ANRBT #4: Does the Scale of the Features Matter?

Let's say that temperature was measured in Fahrenheit, rather than Celsius. How would that affect the model?

In [96]:
# Create a new column for Fahrenheit temperature.


#### Rebuild the `LinearRegression` from above using the `temp_F` features instead.

In [97]:
# Create X and y.


# Instantiate and fit.


# Print the coefficients.


#### Convert 25 degrees Celsius to Fahrenheit.

#### Predict rentals for 77 degrees Fahrenheit.

#### Compare with the prediction for 25 degrees Celsius.

In [98]:
# Re-set up the model from earlier

# Examine the prediction


**Conclusion:** The scale of the features is irrelevant for linear regression models. When changing the scale, we simply change our interpretation of the coefficients and the intercept.

# Conclusions and Takeaways:
* Machine learning, aka statistical modeling, is an important and vast topic we'll spend much of our course exploring. It comes in many varieties for different situations.
* Linear regression is a **supervised "white-box" regression model**.
* As far as machine learning goes, OLS is very simple, yet still powerful.
* One of its more enticing feature is that its **coefficients are interpretable!**
* In the midst of all this, we learned how to handle dummy variables, something important for all of the models we'll learn about.